# CXR-Intel: Multi-Modal Chest X-Ray Intelligence System
**GPU required** | **Add MIMIC dataset via right panel before running**

Steps:
1. Right panel → Add Input → search `simhadrisadaram/mimic-cxr-dataset` → Add
2. Add-ons → Secrets → add `HF_TOKEN` (your Hugging Face token)
3. Settings → Accelerator → GPU T4
4. Run All

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.47.1
!pip install -q colpali-engine
!pip install -q faiss-cpu
!pip install -q rouge-score nltk bert-score
!pip install -q open_clip_torch sentence-transformers
!pip install -q python-dotenv accelerate bitsandbytes pyyaml
print("All packages installed!")

## Cell 2 — Setup + Clone Project from GitHub

In [ ]:
import os, shutil, subprocess

# Load HF token
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN loaded')
except Exception as e:
    print(f'HF_TOKEN not in secrets: {e}')
    os.environ['HF_TOKEN'] = ''

os.environ['USE_MOCK_MODE'] = 'false'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['SAMPLE_SIZE'] = '300'

GITHUB_REPO = 'https://github.com/HAMZA2FEKRY/cxr-intel.git'
WORK_DIR = '/kaggle/working/cxr-intel'

if os.path.exists(WORK_DIR):
    print('Repo found, pulling latest...')
    r = subprocess.run(['git','-C',WORK_DIR,'pull'], capture_output=True, text=True)
    print(r.stdout or 'Already up to date')
else:
    print('Cloning from GitHub...')
    r = subprocess.run(['git','clone',GITHUB_REPO,WORK_DIR], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed: {r.stderr}')

os.chdir(WORK_DIR)
print('Working dir:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))

## Cell 3 — Create config.yaml

In [ ]:
import yaml

config = {
    'dataset': {
        'raw_dir': 'data/raw/mimic_cxr',
        'raw_train_csv': 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv',
        'processed_dir': 'data/processed',
        'sample_size': 300,
        'sample_metadata_path': 'data/samples/sample_metadata.csv'
    },
    'columns': {
        'report_candidates': ['text','report','findings','impression'],
        'image_candidates': ['image_path','path','filename','image','dicom_id'],
        'id_candidates': ['image_id','subject_id','study_id','dicom_id','id']
    },
    'rag': {'top_k': 3, 'index_dir': 'rag/indexes'},
    'models': {
        'medgemma_model_id': 'google/medgemma-1.5-4b-it',
        'use_mock_mode': False
    },
    'app': {'title': 'CXR-Intel'}
}

os.makedirs('configs', exist_ok=True)
with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('config.yaml written')
print(open('configs/config.yaml').read())

## Cell 4 — Link MIMIC CXR Dataset

In [ ]:
import shutil

# Auto-find MIMIC dataset under /kaggle/input/
print('Scanning /kaggle/input/ ...')
try:
    available = os.listdir('/kaggle/input')
    print('Available datasets:', available)
except:
    available = []
    print('No datasets found')

MIMIC_INPUT = None
for d in available:
    if 'mimic' in d.lower() or 'cxr' in d.lower():
        MIMIC_INPUT = f'/kaggle/input/{d}'
        print(f'Found MIMIC at: {MIMIC_INPUT}')
        break

if MIMIC_INPUT is None:
    print('ERROR: MIMIC dataset not attached!')
    print('Fix: Right panel -> Add Input -> simhadrisadaram/mimic-cxr-dataset')
else:
    # Show first 15 files
    shown = 0
    for root, dirs, files in os.walk(MIMIC_INPUT):
        for f in files:
            print(' ', os.path.join(root, f))
            shown += 1
            if shown >= 15: break
        if shown >= 15: break

    # Copy any CSV
    os.makedirs('data/raw/mimic_cxr', exist_ok=True)
    for root, dirs, files in os.walk(MIMIC_INPUT):
        for f in files:
            if f.endswith('.csv'):
                src = os.path.join(root, f)
                dst = 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv'
                shutil.copy2(src, dst)
                print(f'CSV: {f} -> {dst}')
                break
        else: continue
        break

    # Copy images (max 300)
    os.makedirs('data/samples/images', exist_ok=True)
    n = 0
    for root, dirs, files in os.walk(MIMIC_INPUT):
        for f in files:
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                shutil.copy2(os.path.join(root,f),
                             os.path.join('data/samples/images', f))
                n += 1
            if n >= 300: break
        if n >= 300: break
    print(f'Images copied: {n}')
    print('Data setup complete!')

## Cell 5 — Prepare Dataset + Generate QA Pairs

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

print('--- Inspect Dataset ---')
!python scripts/inspect_dataset.py

print('--- Prepare Dataset ---')
!python scripts/prepare_dataset.py

print('--- Create Sample ---')
!python scripts/create_sample.py

print('--- Generate QA ---')
!python scripts/generate_qa_dataset.py

print('Data preparation complete!')

## Cell 6 — Load Models + Build Indexes

In [ ]:
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print('--- Build CLIP Index ---')
!python rag/build_index.py --model clip

print('--- Build ColPali Index ---')
!python rag/build_index.py --model colpali

print('Indexes built!')

## Cell 7 — Run Report Generation (MedGemma)

In [ ]:
print('Running report generation...')
!python scripts/run_report_generation.py

import pandas as pd
df = pd.read_csv('results/report_generation_results.csv')
print(f'Generated {len(df)} reports')
print('Sample:', df['generated_report'].iloc[0][:300])

## Cell 8 — Run QA Pipeline

In [ ]:
print('Running QA pipeline...')
!python scripts/run_qa_pipeline.py

import pandas as pd
df_qa = pd.read_csv('results/qa_results.csv')
print(f'Answered {len(df_qa)} questions')
row = df_qa.iloc[0]
print(f'Q: {row["question"]}')
print(f'A: {row["clip_answer"][:150]}')

## Cell 9 — Evaluate + Compare Models

In [ ]:
!python evaluation/evaluate_reports.py
!python evaluation/evaluate_retrieval.py
!python evaluation/evaluate_qa.py
!python evaluation/compare_models.py

import json
for fname in ['report_generation_metrics.json','retrieval_metrics.json','qa_metrics.json']:
    fpath = f'results/{fname}'
    if os.path.exists(fpath):
        print(f'\n{fname}:')
        print(json.dumps(json.load(open(fpath)), indent=2))

## Cell 10 — Save Results

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/cxr_results','zip', WORK_DIR,'results')
shutil.make_archive('/kaggle/working/cxr_indexes','zip', WORK_DIR,'rag/indexes')
print('Download from Output tab:')
print('  cxr_results.zip')
print('  cxr_indexes.zip')

## Cell 11 — Visual Results Summary

In [ ]:
import matplotlib.pyplot as plt, json

retrieval = json.load(open('results/retrieval_metrics.json'))
qa        = json.load(open('results/qa_metrics.json'))
report    = json.load(open('results/report_generation_metrics.json'))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('CXR-Intel Model Comparison', fontsize=14, fontweight='bold')

ax = axes[0]
ks = ['Retrieval@1','Retrieval@3','Retrieval@5']
x = range(len(ks))
ax.bar([i-.2 for i in x],[retrieval['clip'][k] for k in ks],.4,label='CLIP',color='steelblue')
ax.bar([i+.2 for i in x],[retrieval['colpali'][k] for k in ks],.4,label='ColPali',color='coral')
ax.set_title('Retrieval Performance'); ax.set_xticks(list(x))
ax.set_xticklabels(['@1','@3','@5']); ax.legend(); ax.set_ylim(0,1)

ax = axes[1]
vals = [report['rougeL'], report['bleu']]
ax.bar(['ROUGE-L','BLEU'], vals, color=['mediumseagreen','mediumpurple'])
ax.set_title('Report Generation (MedGemma)'); ax.set_ylim(0,1)
for i,v in enumerate(vals): ax.text(i, v+.01, f'{v:.3f}', ha='center')

ax = axes[2]
vals2 = [qa['clip_similarity'], qa['colpali_similarity']]
ax.bar(['CLIP+MedGemma','ColPali+MedGemma'], vals2, color=['steelblue','coral'])
ax.set_title('QA Similarity'); ax.set_ylim(0,1)
for i,v in enumerate(vals2): ax.text(i, v+.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.savefig('results/comparison.png', dpi=150, bbox_inches='tight')
plt.show()